In [ ]:
import pandas as pd
import torch 
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision 
import torchvision.transforms as transforms
import numpy as np

import matplotlib.pyplot as plt
from PIL import Image

import requests
import json
import ast
import os

import difflib
import random
from tqdm import tqdm

https://www.kaggle.com/competitions/understanding_cloud_organization/

In [ ]:
device = 'mps'

### Load Clouds Dataframe

In [ ]:
DATASET_PATH = "/Users/emulie/Downloads/understanding_cloud_organization/"

In [ ]:
df_train = pd.read_csv(DATASET_PATH + 'train.csv')

In [ ]:
df_train.head()

In [ ]:
df_train.shape

In [ ]:
torch.manual_seed(42)

In [ ]:
class CloudDataset(Dataset):
    
    def __init__(self, dataset_path: str, split: str, train_split: float = 0.8):
        self.dataset_path = dataset_path
        self.split = split
        self.train_split = train_split
        
        self.image_files, self.labels, self.encoded_pixels = self._load_image_files()
        
    def _load_image_files(self):
        # read csv file
        df_train = pd.read_csv(self.dataset_path + 'train.csv')
        
        # read image metadata
        n = df_train.shape[0]
        e_train = int(n * self.train_split)
        df_train['label'] = df_train['Image_Label'].apply(lambda x: x.split('_')[1])
        df_train['img_file'] = df_train['Image_Label'].apply(lambda x: x.split('_')[0])
        
        image_files = df_train['img_file'].tolist()
        labels = df_train['label'].tolist()
        encoded_pixels = [pixels if not pd.isna(pixels) else [] for pixels in df_train['EncodedPixels'].tolist()]
        
        if self.split == 'train':
            return image_files[:e_train], labels[:e_train], encoded_pixels[:e_train]
        elif self.split == 'test':
            return image_files[e_train:], labels[e_train:], encoded_pixels[e_train:]
        else:
            raise ValueError(f"Split not implemented. Please select 'train' or 'test'")
            
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        sub_directory = 'train_images/' if self.split == 'train' else 'test_images/'
        image = Image.open(self.dataset_path + sub_directory + self.image_files[idx])
        img_array = np.asarray(image)
        
        return {'image': img_array, 'label': self.labels[idx], 
                'encoded_pixels': self.encoded_pixels[idx]}


In [ ]:
train_dataset = CloudDataset(DATASET_PATH, 'train', train_split=0.8)
test_dataset = CloudDataset(DATASET_PATH, 'test', train_split=0.8)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=True)

In [ ]:
batch = next(iter(train_loader))

In [ ]:
batch['image'].size() # # torch.Size([1, 1400, 2100, 3])

### Visualize Cloud

In [ ]:
batch = next(iter(train_loader))
# batch

In [ ]:
shape = (1400, 2100)

# note: encoded_pixels => pair (start_pixel, length)
# RLE is 1-based so start = number[i] - 1

def rle_decode(encoded_pixels, shape=shape):
    height, width = shape[0], shape[1]
    encoded_pixels = list(map(int, encoded_pixels.split()))

    mask =  np.zeros(height * width, dtype=np.uint8)
    for i in range(0, len(encoded_pixels), 2):
        start, length = encoded_pixels[i] - 1, encoded_pixels[i+1]
        mask[start:start+length] = 1
    mask = mask.reshape((width, height)).T
    return mask
    

In [ ]:
mask = rle_decode(batch['encoded_pixels'][0])
plt.imshow(batch['image'][0])
plt.imshow(mask, alpha=0.25)
plt.title(batch['label'][0])
plt.show()